In [ ]:

# CONFIGURAÇÕES INICIAIS DAS ANÁLISES (PRESENTES EM TODOS OS SCRIPTS)
# IMPORTAR BIBLIOTECAS ---
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# CAMINHOS ---
SCRIPT_DIR = Path(__file__).resolve().parent        # caminho desse script
ANALYTICS_DIR = SCRIPT_DIR.parent                   # pasta desse script

# PARA IMPORTAR FUNÇÕES DE EXTRAIR CSV ---
if str(ANALYTICS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYTICS_DIR))
from utils.export_utils import exportar_csv


# ROOT DO PROJETO ---
PROJECT_ROOT = Path(__file__).resolve().parents[3]

# ROOT DOS OUTPUTS ---
OUTPUT_DIR = (PROJECT_ROOT/ "scripts"/ "Analytics"/ "outputs"/ "gold_01")
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# SPARK ---
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# PREPARAÇÃO PARA ANALISAR GOLD 01 - Como está estruturado o mercado brasileiro de Dados? ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CAMINHO DO ARQUIVO ---
caminho_gold_01 = (PROJECT_ROOT/ "Gold"/ "perguntas_negocio"/ "gold_01_estrutura_mercado")

# BUSCA CSVs GERADOS PELO SPARK NA CRIAÇÃO DA GOLD ---
arquivos_gold_01 = [
    str(arquivo)
    for arquivo in caminho_gold_01.glob("part-*.csv")
]
print("Arquivos encontrados:")
print(arquivos_gold_01)

# CARREGAR GOLD 01 ---
df_estrutura = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_01)
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# INSPEÇÃO INICIAL - PARA IDENTIFICAR O QUE EXISTE NO df_estrutura ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# OLHAR AS COLUNAS E IDENTIFICAR QTD DE LINHAS ---
df_estrutura.show(truncate=False)
df_estrutura.printSchema()
print("Quantidade de linhas:", df_estrutura.count())
print("Colunas:", df_estrutura.columns)

# VERIFICAR AS VARIÁVEIS PRESENTES NA GOLD 01
df_estrutura.select("variavel").distinct().show(truncate=False)

# VERIFICAR AS EDIÇÕES EXISTENTES
df_estrutura.select("edicao").distinct().show(truncate=False)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# INÍCIO DAS ANÁLISES ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CARGOS ---
# ANÁLISE INICIAL DA DIMENSÃO ---
df_cargos = (df_estrutura
             .filter(F.col("variavel") == "cargo_atual")
             .orderBy("edicao", F.desc("pct_na_dimensao"))
            )
df_cargos.show(100,truncate=False)
print("Qtd linhas:", df_cargos.count())
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# TOP 5 CARGOS - ANÁLISE EXPLORATÓRIA ---
janela_ranking_original = (Window.partitionBy("edicao")
                           .orderBy(F.desc("pct_na_dimensao"))
                            )

top_cargos = (df_cargos.withColumn("ranking",F.row_number().over(janela_ranking_original))
    .filter(F.col("ranking") <= 5)
    .select("edicao","valor","contagem","pct_na_dimensao","ranking")
    .orderBy("edicao","ranking")
)
top_cargos.show(100,truncate=False)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# COMPARATIVO ENTRE OS ANOS ---
comparativo_cargos = (top_cargos
    .groupBy("valor")
    .pivot("edicao")
    .agg(F.first("pct_na_dimensao"))
    .orderBy(F.desc("2025-2026"))
)
comparativo_cargos.show(truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
O comparativo exploratório dos principais cargos revelou diferenças
na nomenclatura das categorias entre as edições.

Foram identificados valores nulos no comparativo porque determinados
cargos não aparecem com exatamente a mesma nomenclatura em todos os
períodos.

Por exemplo, Engenharia e Arquitetura de Dados estavam agrupadas em
uma única opção em 2023-2024 e passaram a aparecer de forma diferente
nas edições seguintes.

Por esse motivo, foi necessária a harmonização da taxonomia antes da
realização das comparações históricas.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# VERIFICAR QUAIS CARGOS EXISTEM POR ANO DE PESQUISA ---
(df_cargos
    .select("edicao","valor")
    .distinct()
    .orderBy("edicao","valor")
    .show(200,truncate=False)
)

# HARMONIZAR OS CARGOS ---
df_cargos_taxonomia = (df_cargos
    .withColumn("cargo_harmonizado",
        F.when(
            F.col("valor").isin(
                (
                    "Engenheiro de Dados/Arquiteto de Dados/"
                    "Data Engineer/Data Architect"
                ),
                (
                    "Engenheiro de Dados/"
                    "Data Engineer/Data Architect"
                ),
                "Arquiteto de Dados/Data Architect"
            ),
            "Engenharia e Arquitetura de Dados"
        )
        .otherwise(
            F.col("valor")
        )
    )
)

# VALIDAR A HARMONIZAÇÃO ---
(df_cargos_taxonomia
    .select("edicao","valor","cargo_harmonizado")
    .distinct()
    .orderBy("edicao","cargo_harmonizado")
    .show(200,truncate=False)
)

# CONSOLIDAR CARGOS PÓS HARMONIZAÇÃO ---
df_cargos_consolidados = (df_cargos_taxonomia
    .groupBy("edicao","cargo_harmonizado")
    .agg(F.sum("contagem").alias("contagem"))
)

# RECÁLCULO DOS PERCENTUAIS ---
janela_edicao = (Window.partitionBy("edicao"))

df_cargos_consolidados = (df_cargos_consolidados
    .withColumn("total_edicao",F.sum("contagem").over(janela_edicao))
    .withColumn("pct_na_dimensao",F.round((F.col("contagem")/ F.col("total_edicao"))* 100,2))
)

# RESULTADO DOS CARGOS HARMONIZADOS ---
(df_cargos_consolidados
    .orderBy("edicao",F.desc("pct_na_dimensao"))
    .show(200,truncate=False)
)

# VALIDAÇÃO DOS PERCENTUAIS ---
(df_cargos_consolidados
    .groupBy("edicao")
    .agg(F.sum("contagem").alias("total_respondentes"),F.round(F.sum("pct_na_dimensao"),2).alias("soma_percentual"))
    .orderBy("edicao")
    .show(truncate=False)
)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A validação confirma que os percentuais recalculados após a
harmonização permanecem consistentes, totalizando aproximadamente
100% em todas as edições.

A pequena diferença observada (99,99% ou 100,01%) ocorre em função
do arredondamento dos percentuais para duas casas decimais.

Também é importante observar que o número de respondentes varia
entre as edições:

- 2023-2024: 3.857 respondentes
- 2024-2025: 3.818 respondentes
- 2025-2026: 2.501 respondentes

Por esse motivo, as comparações históricas serão realizadas
principalmente por participação percentual, e não pela contagem
absoluta de respondentes.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# TOP 5 CARGOS APÓS HARMONIZAÇÃO ---
janela_ranking = (Window.partitionBy("edicao")
    .orderBy(F.desc("pct_na_dimensao"))
)

top_cargos_harmonizados = (df_cargos_consolidados
    .withColumn("ranking",F.row_number().over(janela_ranking))
    .filter(F.col("ranking") <= 5)
    .select("edicao","cargo_harmonizado","contagem","pct_na_dimensao","ranking")
    .orderBy("edicao","ranking")
)
top_cargos_harmonizados.show(100,truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Após a harmonização da taxonomia, os cinco cargos com maior
representatividade são os mesmos nas três edições analisadas:

- Analista de Dados/Data Analyst
- Cientista de Dados/Data Scientist
- Engenharia e Arquitetura de Dados
- Analista de BI/BI Analyst
- Outra Opção

Analista de Dados/Data Analyst permanece como o cargo mais
representativo em todas as edições, com participação próxima
de um quarto da amostra.

Entre 2023-2024 e 2024-2025, Cientista de Dados ocupa a segunda
posição e Engenharia e Arquitetura de Dados a terceira.

Em 2025-2026 ocorre uma inversão entre essas duas posições:
Engenharia e Arquitetura de Dados passa para 17,19% e Cientista
de Dados fica com 16,95%.

Analista de BI permanece na quarta posição, mas apresenta redução
contínua de participação ao longo das três edições.

A categoria Outra Opção permanece na quinta posição e aumenta sua
participação na edição mais recente.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# IDENTIFICAÇÃO DOS CARGOS COMPARÁVEIS ---
cargos_comparabilidade = (df_cargos_consolidados
    .groupBy("cargo_harmonizado")
    .agg(F.countDistinct("edicao").alias("qtd_edicoes"))
    .withColumn("comparavel_historicamente",F.col("qtd_edicoes") == 3)
    .orderBy(F.desc("comparavel_historicamente"),"cargo_harmonizado")
)
cargos_comparabilidade.show(100,truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A análise de comparabilidade identificou 14 categorias de cargos
presentes nas três edições da pesquisa.

Esses cargos serão utilizados na comparação histórica por possuírem
observações em todos os períodos analisados.

Três categorias aparecem somente em 2023-2024 e, portanto, não serão
utilizadas na análise de evolução entre as três edições:

- Analista de Inteligência de Mercado/Market Intelligence
- DBA/Administrador de Banco de Dados
- Economista

A exclusão dessas categorias do comparativo histórico evita interpretar
a ausência de uma opção nas pesquisas seguintes como uma redução real
de participação.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CARGOS PRESENTES NAS TRÊS EDIÇÕES ---
cargos_comparaveis = (cargos_comparabilidade
    .filter(F.col("comparavel_historicamente") == True)
    .select("cargo_harmonizado")
)

# BASE HISTÓRICA COM CARGOS COMPARÁVEIS ---
df_historico_cargos = (df_cargos_consolidados
    .join(cargos_comparaveis,on="cargo_harmonizado",how="inner")
)

# COMPARATIVO HISTÓRICO ---
comparativo_historico = (df_historico_cargos
    .groupBy("cargo_harmonizado")
    .pivot("edicao",
        [
            "2023-2024",
            "2024-2025",
            "2025-2026"
        ]
    )
    .agg(F.first("pct_na_dimensao"))
)
comparativo_historico.show(100,truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
O comparativo histórico evidencia comportamentos diferentes entre
as categorias de cargos.

Analista de Dados/Data Analyst mantém participação relativamente
estável e permanece como o principal grupo nas três edições,
variando entre aproximadamente 23,5% e 25,1%.

Engenharia e Arquitetura de Dados também apresenta estabilidade,
permanecendo próxima de 17% em todo o período.

Cientista de Dados/Data Scientist mantém participação próxima de
17% a 18%, embora apresente redução na edição mais recente.

Em contraste, Analista de BI/BI Analyst apresenta redução contínua,
passando de 13,12% em 2023-2024 para 8,60% em 2025-2026.

Algumas categorias apresentam trajetória de aumento, especialmente
Engenheiro de Machine Learning/ML Engineer/AI Engineer,
Analytics Engineer e Desenvolvimento/Engenharia de Software.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# VARIAÇÃO ENTRE PRIMEIRA E ÚLTIMA EDIÇÃO ---
comparativo_historico = (comparativo_historico
    .withColumn("variacao_pp",F.round((F.col("2025-2026")- F.col("2023-2024")),2))
    .orderBy(F.desc("variacao_pp"))
)
comparativo_historico.show(100,truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A variação em pontos percentuais entre 2023-2024 e 2025-2026 permite
identificar quais categorias ganharam ou perderam participação relativa
na amostra.

Os maiores aumentos de participação foram observados em:

- Engenheiro de Machine Learning/ML Engineer/AI Engineer: +2,30 p.p.
- Analytics Engineer: +1,80 p.p.
- Desenvolvedor/Engenheiro de Software/Analista de Sistemas: +1,52 p.p.
- Outra Opção: +1,45 p.p.

A maior redução foi observada em Analista de BI/BI Analyst,
com queda de 4,52 pontos percentuais.

Os resultados representam alterações na composição dos respondentes
das pesquisas e não devem ser interpretados isoladamente como
crescimento ou redução do número de profissionais no mercado de trabalho.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# ESTRUTURA ATUAL DO MERCADO (EDIÇÃO 2025-2026) ---
estrutura_atual = (df_cargos_consolidados
    .filter(F.col("edicao") == "2025-2026")
    .select("cargo_harmonizado","contagem","pct_na_dimensao")
    .orderBy(F.desc("pct_na_dimensao"))
)
estrutura_atual.show(100,truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Na edição mais recente (2025-2026), Analista de Dados/Data Analyst
é o cargo com maior participação na amostra, representando 23,95%
dos respondentes.

Na sequência aparecem:

- Engenharia e Arquitetura de Dados: 17,19%
- Cientista de Dados/Data Scientist: 16,95%
- Analista de BI/BI Analyst: 8,60%
- Outra Opção: 8,24%

Os resultados mostram uma concentração relevante da amostra nas
funções diretamente relacionadas à análise, ciência e engenharia
de dados.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CONCENTRAÇÃO DOS 3 PRINCIPAIS CARGOS ---
top3_atual = (estrutura_atual
    .limit(3)
    .agg(F.round(F.sum("pct_na_dimensao"),2).alias("participacao_top3"))
)
top3_atual.show(truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Os três cargos com maior participação em 2025-2026 concentram
58,09% dos respondentes da dimensão de cargos.

Isso significa que mais da metade da amostra está concentrada em
três grandes grupos:

- Analista de Dados/Data Analyst
- Engenharia e Arquitetura de Dados
- Cientista de Dados/Data Scientist

Esse resultado evidencia uma forte concentração da amostra nas
funções centrais da área de Dados.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# MAIORES CRESCIMENTOS ---
maiores_crescimentos = (comparativo_historico
    .filter(F.col("variacao_pp") > 0)
    .orderBy(F.desc("variacao_pp"))
)
maiores_crescimentos.show(100,truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Entre os cargos comparáveis historicamente, Engenheiro de Machine
Learning/ML Engineer/AI Engineer apresenta o maior aumento de
participação entre a primeira e a última edição, passando de
1,94% para 4,24%, uma variação de +2,30 pontos percentuais.

Analytics Engineer apresenta o segundo maior aumento, de 3,60%
para 5,40% (+1,80 p.p.).

Desenvolvedor/Engenheiro de Software/Analista de Sistemas também
amplia sua participação, passando de 2,72% para 4,24%
(+1,52 p.p.).

Esses resultados indicam aumento da representatividade dessas
categorias entre os respondentes, mas não permitem concluir,
isoladamente, que houve crescimento equivalente desses cargos
no mercado de trabalho brasileiro.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# MAIORES QUEDAS ---
maiores_quedas = (comparativo_historico
    .filter(F.col("variacao_pp") < 0)
    .orderBy("variacao_pp")
)
maiores_quedas.show(100,truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Analista de BI/BI Analyst apresenta a maior redução de participação
entre as categorias historicamente comparáveis, passando de 13,12%
em 2023-2024 para 8,60% em 2025-2026, uma variação de -4,52 pontos
percentuais.

A redução é significativamente maior do que a observada nas demais
categorias com variação negativa.

Cientista de Dados/Data Scientist apresenta redução de -0,86 p.p.,
enquanto Data Product Manager/Product Manager registra -0,78 p.p.
e Engenharia e Arquitetura de Dados -0,54 p.p.

Portanto, entre os cargos analisados, Analista de BI é a categoria
que apresenta a mudança negativa mais expressiva na composição
da amostra ao longo do período.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# EXPORTAÇÃO DOS RESULTADOS PARA VISUALIZAÇÃO ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# ESTRUTURA ATUAL.CSV ---
exportar_csv(estrutura_atual,OUTPUT_DIR,"cargos_estrutura_atual.csv")

# HISTÓRICO.CSV ---
exportar_csv(comparativo_historico, OUTPUT_DIR,"cargos_historico.csv")